In [ ]:
# =========================
# nb_ingest_brown_shipley
# =========================

# ---------- Parameters ----------
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by nb_run_all in real run

dfm_id = "brown_shipley"
dfm_name = "Brown Shipley"

# ---------- Imports ----------
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
import re

# ---------- Paths ----------
landing_path = f"Files/landing/period={period}/dfm={dfm_id}/source/"
fx_path = "Files/config/fx_rates.csv"

# ---------- Helpers ----------
def q(s: str) -> str:
    return (s or "").replace("'", "''")

def norm(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")

def dec_col(c, p=28, s=10):
    return F.col(c).cast(f"decimal({p},{s})")

def write_audit(files_processed, rows_ingested, parse_errors_count, drift_events_count, status):
    spark.sql(f"""
        INSERT INTO run_audit_log
        (run_id, period, dfm_id, files_processed, rows_ingested, parse_errors_count, drift_events_count, status, started_at, completed_at)
        VALUES
        ('{q(run_id)}','{q(period)}','{q(dfm_id)}',{int(files_processed)},{int(rows_ingested)},{int(parse_errors_count)},{int(drift_events_count)},'{q(status)}',current_timestamp(),current_timestamp())
    """)

def write_parse_errors(df_err):
    cols = set(df_err.columns)
    out = df_err

    if "source_file" not in cols:
        out = out.withColumn("source_file", F.lit("UNKNOWN"))
    if "row_number" not in cols:
        if "source_row_id" in cols:
            out = out.withColumn("row_number", F.regexp_extract(F.col("source_row_id").cast("string"), r"(\\d+)$", 1).cast("bigint"))
        else:
            out = out.withColumn("row_number", F.lit(None).cast("bigint"))
    if "column_name" not in cols:
        out = out.withColumn("column_name", F.lit(None).cast("string"))
    if "raw_value" not in cols:
        out = out.withColumn("raw_value", F.lit(None).cast("string"))
    if "error_type" not in cols:
        out = out.withColumn("error_type", F.lit("PARSE_ERROR"))
    if "error_message" not in cols:
        out = out.withColumn("error_message", F.lit("Unknown parse error"))

    (
        out.withColumn("period", F.lit(period))
           .withColumn("run_id", F.lit(run_id))
           .withColumn("dfm_id", F.lit(dfm_id))
           .withColumn("event_ts", F.current_timestamp())
           .select("period","run_id","dfm_id","source_file","row_number","column_name","raw_value","error_type","error_message","event_ts")
           .write.mode("append").saveAsTable("parse_errors")
    )

def euro_to_double(c):
    # "1.234,56" -> "1234.56"
    return F.regexp_replace(
        F.regexp_replace(F.trim(F.col(c).cast("string")), r"\.", ""),
        ",", "."
    ).cast("double")

def detect_header_and_read_csv(path, expected_norm_cols, source_file):
    """
    Robust header detection using pandas skiprows 0..10.
    """
    best_pdf = None
    for skip in range(0, 11):
        try:
            pdf = pd.read_csv(path, skiprows=skip, dtype=str, encoding="utf-8")
            pdf.columns = [norm(c) for c in pdf.columns]
            if all(c in pdf.columns for c in expected_norm_cols):
                best_pdf = pdf
                break
        except Exception:
            continue

    if best_pdf is None:
        return None

    sdf = spark.createDataFrame(best_pdf)
    sdf = sdf.withColumn("source_file", F.lit(source_file))
    return sdf

# ---------- A3: File discovery ----------
try:
    entries = [f for f in mssparkutils.fs.ls(landing_path) if not f.isDir]
except Exception:
    write_audit(0, 0, 0, 0, "NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")

csv_files = [f for f in entries if f.name.lower().endswith(".csv")]
if len(csv_files) == 0:
    write_audit(0, 0, 0, 0, "NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")

# role files from config ground truth
pos_file = next((f for f in csv_files if f.name.lower() == "notification.csv"), None)
cash_file = next((f for f in csv_files if f.name.lower() == "notification - cash.csv"), None)

if pos_file is None and cash_file is None:
    write_audit(len(csv_files), 0, 0, 0, "NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")

# ---------- A4: Parse role files ----------
parse_err_seed = []

pos_expected = ["client_id", "value_date", "isin", "balance", "accrued_interest"]
cash_expected = ["client_id", "accounting_balance"]

df_pos = None
if pos_file is not None:
    df_pos = detect_header_and_read_csv(pos_file.path, pos_expected, pos_file.name)
    if df_pos is None:
        parse_err_seed.append((pos_file.name, None, "header_detection", "", "PARSE_ERROR", "Could not detect valid header for positions file"))

df_cash = None
if cash_file is not None:
    df_cash = detect_header_and_read_csv(cash_file.path, cash_expected, cash_file.name)
    if df_cash is None:
        parse_err_seed.append((cash_file.name, None, "header_detection", "", "PARSE_ERROR", "Could not detect valid header for cash file"))

from pyspark.sql.types import StructType, StructField, StringType, LongType

seed_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("row_number", LongType(), True),
    StructField("column_name", StringType(), True),
    StructField("raw_value", StringType(), True),
    StructField("error_type", StringType(), True),
    StructField("error_message", StringType(), True),
])

seed_df = spark.createDataFrame(parse_err_seed, seed_schema)
write_parse_errors(seed_df)

if df_pos is None and df_cash is None:
    write_audit(len(csv_files), 0, len(parse_err_seed), 0, "FAILED")
    mssparkutils.notebook.exit("FAILED")

# add row ids
if df_pos is not None:
    w = Window.orderBy(F.monotonically_increasing_id())
    df_pos = (
        df_pos.withColumn("_rn", F.row_number().over(w))
              .withColumn("source_row_id_pos", F.concat_ws(":", F.col("source_file"), F.col("_rn").cast("string")))
              .drop("_rn")
    )

if df_cash is not None:
    w = Window.orderBy(F.monotonically_increasing_id())
    df_cash = (
        df_cash.withColumn("_rn", F.row_number().over(w))
               .withColumn("source_row_id_cash", F.concat_ws(":", F.col("source_file"), F.col("_rn").cast("string")))
               .drop("_rn")
    )

# ---------- A5/A6: Map + cast ----------
if df_pos is not None:
    p = (
        df_pos
        .withColumn("policy_id_p", F.col("client_id").cast("string"))
        .withColumn("dfm_policy_id_p", F.col("client_id").cast("string"))
        .withColumn("report_date_p", F.to_date(F.col("value_date")))
        .withColumn("isin_p", F.col("isin").cast("string"))
        .withColumn("security_id_p", F.col("isin").cast("string"))
        .withColumn("bid_value_local_p", euro_to_double("balance"))
        .withColumn("accrued_interest_local_p", euro_to_double("accrued_interest"))
        .withColumn("local_currency_p", F.lit(None).cast("string"))  # may be absent in source
        .withColumn("holding_p", F.lit(None).cast("double"))
        .withColumn("local_bid_price_p", F.lit(None).cast("double"))
    )
else:
    p = spark.createDataFrame([], """
        source_file string, source_row_id_pos string, policy_id_p string, dfm_policy_id_p string,
        report_date_p date, isin_p string, security_id_p string, bid_value_local_p double,
        accrued_interest_local_p double, local_currency_p string, holding_p double, local_bid_price_p double
    """)

if df_cash is not None:
    c = (
        df_cash
        .withColumn("policy_id_c", F.col("client_id").cast("string"))
        .withColumn("cash_value_local_c", euro_to_double("accounting_balance"))
    )
else:
    c = spark.createDataFrame([], "source_file string, source_row_id_cash string, policy_id_c string, cash_value_local_c double")

joined = p.alias("p").join(c.alias("c"), F.col("p.policy_id_p") == F.col("c.policy_id_c"), "full_outer")

d = (
    joined
    .withColumn("policy_id", F.coalesce(F.col("p.policy_id_p"), F.col("c.policy_id_c")))
    .withColumn("dfm_policy_id", F.coalesce(F.col("p.dfm_policy_id_p"), F.col("c.policy_id_c")))
    .withColumn("report_date", F.col("p.report_date_p"))
    .withColumn("isin", F.col("p.isin_p"))
    .withColumn("security_id", F.col("p.security_id_p"))
    .withColumn("holding_d", F.col("p.holding_p"))
    .withColumn("local_bid_price_d", F.col("p.local_bid_price_p"))
    .withColumn("bid_value_local_d", F.col("p.bid_value_local_p"))
    .withColumn("accrued_interest_local_d", F.col("p.accrued_interest_local_p"))
    .withColumn("cash_value_local_d", F.col("c.cash_value_local_c"))
    .withColumn("local_currency_raw", F.col("p.local_currency_p"))
    .withColumn("source_file_out", F.coalesce(F.col("p.source_file"), F.col("c.source_file")))
    .withColumn("source_row_id_raw", F.coalesce(F.col("p.source_row_id_pos"), F.col("c.source_row_id_cash")))
)

# currency rules
d = d.withColumn(
    "local_currency",
    F.when(F.col("local_currency_raw").isNull() | (F.trim(F.col("local_currency_raw")) == ""), F.lit("GBP"))
     .otherwise(F.upper(F.trim(F.col("local_currency_raw"))))
)

fx = (
    spark.read.option("header", True).csv(fx_path)
    .withColumn("currency_u", F.upper(F.trim(F.col("currency"))))
    .withColumn("rate_to_gbp_d", F.col("rate_to_gbp").cast("double"))
    .select("currency_u", "rate_to_gbp_d")
)

d = d.join(fx, F.col("local_currency") == F.col("currency_u"), "left")

d = (
    d
    .withColumn(
        "fx_rate_out",
        F.when(F.col("local_currency") == "GBP", F.lit(1.0))
         .when(F.col("rate_to_gbp_d").isNotNull(), F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "bid_value_gbp_d",
        F.when(F.col("bid_value_local_d").isNull(), F.lit(None).cast("double"))
         .when(F.col("local_currency") == "GBP", F.col("bid_value_local_d"))
         .when(F.col("rate_to_gbp_d").isNotNull(), F.col("bid_value_local_d") * F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "cash_value_gbp_d",
        F.when(F.col("cash_value_local_d").isNull(), F.lit(0.0))
         .when(F.col("local_currency") == "GBP", F.col("cash_value_local_d"))
         .when(F.col("rate_to_gbp_d").isNotNull(), F.col("cash_value_local_d") * F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "accrued_interest_gbp_d",
        F.when(F.col("accrued_interest_local_d").isNull(), F.lit(0.0))
         .when(F.col("local_currency") == "GBP", F.col("accrued_interest_local_d"))
         .when(F.col("rate_to_gbp_d").isNotNull(), F.col("accrued_interest_local_d") * F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
)

# flags (Brown Shipley-specific only)
d = (
    d
    .withColumn(
        "f_currency_assumed",
        F.when(F.col("local_currency_raw").isNull() | (F.trim(F.col("local_currency_raw")) == ""), F.lit("CURRENCY_ASSUMED_GBP"))
    )
    .withColumn(
        "f_fx_not_available",
        F.when((F.col("local_currency") != "GBP") & F.col("rate_to_gbp_d").isNull(), F.lit("FX_NOT_AVAILABLE"))
    )
    .withColumn(
        "data_quality_flags",
        F.expr("filter(array(f_currency_assumed, f_fx_not_available), x -> x is not null)")
    )
)

# ---------- Quality guards ----------
bad_required = (
    d.filter(F.col("policy_id").isNull())
     .select(
         F.col("source_file_out").alias("source_file"),
         F.coalesce(F.col("source_row_id_raw"), F.lit("UNKNOWN")).alias("source_row_id"),
         F.lit("Required field missing: policy_id").alias("error_message"),
         F.lit("").alias("raw_value")
     )
)

bad_count = bad_required.count()
if bad_count > 0:
    write_parse_errors(
        bad_required
        .withColumn("column_name", F.lit("policy_id"))
        .withColumn("error_type", F.lit("MISSING_REQUIRED"))
    )

warn_report_date = (
    d.filter(F.col("policy_id").isNotNull() & F.col("report_date").isNull())
     .select(
         F.col("source_file_out").alias("source_file"),
         F.coalesce(F.col("source_row_id_raw"), F.lit("UNKNOWN")).alias("source_row_id"),
         F.lit("report_date is null (allowed in PoC for some cash-only rows)").alias("error_message"),
         F.lit("").alias("raw_value")
     )
)

warn_count = warn_report_date.count()
if warn_count > 0:
    write_parse_errors(
        warn_report_date
        .withColumn("column_name", F.lit("report_date"))
        .withColumn("error_type", F.lit("WARNING"))
    )

good = d.filter(F.col("policy_id").isNotNull())

# ---------- A7: source_row_id hash ----------
good = good.withColumn(
    "source_row_id",
    F.sha2(
        F.concat_ws(
            "|",
            F.lit(period),
            F.lit(dfm_id),
            F.col("policy_id"),
            F.coalesce(F.col("security_id"), F.lit("")),
            F.coalesce(F.col("report_date").cast("string"), F.lit("")),
            F.coalesce(F.col("bid_value_local_d").cast("string"), F.lit("")),
            F.coalesce(F.col("cash_value_local_d").cast("string"), F.lit("")),
            F.col("source_file_out"),
            F.coalesce(F.col("source_row_id_raw"), F.lit(""))
        ),
        256
    )
)

good_clean = good.select(
    F.col("policy_id"),
    F.col("report_date"),
    F.col("source_row_id"),
    F.col("source_file_out").alias("source_file"),
    F.col("dfm_policy_id"),
    F.col("security_id").alias("security_id"),
    F.col("isin").alias("isin"),
    F.col("holding_d").alias("holding"),
    F.col("local_bid_price_d").alias("local_bid_price"),
    F.col("local_currency").alias("local_currency"),
    F.col("fx_rate_out").alias("fx_rate"),
    F.col("cash_value_gbp_d").alias("cash_value_gbp"),
    F.col("bid_value_gbp_d").alias("bid_value_gbp"),
    F.col("accrued_interest_gbp_d").alias("accrued_interest_gbp"),
    F.col("data_quality_flags")
)
# ---------- Canonical projection ----------
canonical = (
    good_clean
    .withColumn("period", F.lit(period))
    .withColumn("run_id", F.lit(run_id))
    .withColumn("dfm_id", F.lit(dfm_id))
    .withColumn("dfm_name", F.lit(dfm_name))
    .withColumn("source_sheet", F.lit(None).cast("string"))
    .withColumn("policy_id_type", F.lit("DFM"))
    .withColumn("other_security_id", F.lit(None).cast("string"))
    .withColumn("id_type", F.when(F.col("isin").isNotNull(), F.lit("ISIN")).otherwise(F.lit(None).cast("string")))
    .withColumn("asset_name", F.lit(None).cast("string"))
    .withColumn("ingested_at", F.current_timestamp())
    .select(
        "period",
        "run_id",
        "dfm_id",
        "dfm_name",
        "source_file",
        "source_sheet",
        "source_row_id",
        "policy_id",
        "policy_id_type",
        "dfm_policy_id",
        "security_id",
        "isin",
        "other_security_id",
        "id_type",
        "asset_name",
        dec_col("holding").alias("holding"),
        dec_col("local_bid_price").alias("local_bid_price"),
        "local_currency",
        dec_col("fx_rate").alias("fx_rate"),
        dec_col("cash_value_gbp").alias("cash_value_gbp"),
        dec_col("bid_value_gbp").alias("bid_value_gbp"),
        dec_col("accrued_interest_gbp").alias("accrued_interest_gbp"),
        "report_date",
        "ingested_at",
        "data_quality_flags"
    )
    .dropDuplicates(["run_id", "source_row_id"])
)

# ---------- A8: Write ----------
canonical.write.mode("append").saveAsTable("canonical_holdings")
rows_ingested = canonical.count()

# ---------- A9: Audit ----------
status = "PARTIAL" if (bad_count + warn_count + len(parse_err_seed)) > 0 else "OK"
write_audit(
    files_processed=len(csv_files),
    rows_ingested=rows_ingested,
    parse_errors_count=bad_count + warn_count + len(parse_err_seed),
    drift_events_count=0,
    status=status
)

# ---------- A10: Exit ----------
mssparkutils.notebook.exit(status)